# Walkthrough: data → simulation → features → results
Run from the project root after `python train.py` and `python evaluate.py`. All heavy logic lives in `src/`; this notebook only inspects artefacts.

In [ ]:
import sys, json; sys.path.insert(0, '..')
import os; os.chdir('..')
import pandas as pd
from src.utils.config import load_config
from src.data.dataset import load_processed
cfg = load_config()
dfs, events = load_processed(cfg)
events.groupby(['asset','kind','type']).size()

## Real vs injected signal
Observed = real measurement + `sim_delta_*`; the real ETT measurement is always recoverable.

In [ ]:
df = dfs['ETTm1']
r = events[(events.kind=='failure') & (events.type=='thermal_runaway')].iloc[0]
seg = df.iloc[r.failure_idx-96 : r.failure_idx+24]
ax = (seg.set_index('timestamp')['OT'] - seg.set_index('timestamp')['sim_delta_OT']).plot(label='real OT')
seg.set_index('timestamp')['OT'].plot(ax=ax, label='observed OT'); ax.legend();

## Leakage checks

In [ ]:
print(open('reports/leakage_report.json').read())

## Experiment log (selection on validation columns only)

In [ ]:
res = pd.read_csv('experiments/results.csv')
res.groupby(['family','model'])[['val_pr_auc','pr_auc','f1','event_recall_detectable','false_alarms_per_day']].mean().round(3)

## Final model

In [ ]:
fm = json.load(open('reports/final_metrics.json'))
pd.DataFrame({k: fm[k] for k in ('val','test')}).loc[['precision','recall','f1','roc_auc','pr_auc','event_recall_detectable','lead_time_median_min','false_alarms_per_day']]